In [2]:
# import packages. 
import pandas as pd
import geopandas as gpd
import seaborn as sns
from pathlib import Path
import folium

In [3]:
# file path to import data
root_fldr = Path.cwd().parent
raw_fldr = root_fldr / 'data' / 'raw' 
processed_fldr = root_fldr / 'data' / 'processed'
fire_data_path= processed_fldr / 'fire_by_county_2022_2024.csv'
income_data_path = processed_fldr /'County_Income_CFP_2022_2024.csv'

In [4]:
fire_data = pd.read_csv(fire_data_path)
income_data = pd.read_csv(income_data_path)

income_data.head()

,GEOID,county,year,Median_income,Income_MOE,policies,exposure
0,6001,Alameda,2022,122488,1231,2956,3047740652
1,6003,Alpine,2022,101125,17442,263,179049160
2,6005,Amador,2022,74853,6048,4041,2808434094
3,6007,Butte,2022,66085,2261,3722,2472656045
4,6009,Calaveras,2022,77526,3875,6983,4970383068


In [5]:
#to simplify df for join between income and fire data 
fire_data_clean = fire_data.drop(columns=
    [
    'Unnamed: 0',
    #'county_name',
    #'fire_name',
    #'acres_burned',
    #'year',
    #'total_fire_acres',
    'sqmi_burned',
    #'percent_fire_county',
    'cause_description',
    'fire_length_days',
    #'geometry'
    ]
)
fire_data_clean.head()

,fire_name,total_fire_acres,year,county_name,geometry,acres_burned,percent_fire_county
0,PARK,429603.0,2024,Butte County,MULTIPOLYGON (((-137757.5718037696 224708.7088...,52896.557968,12.312893
1,PARK,429603.0,2024,Tehama County,MULTIPOLYGON (((-162090.1114197266 262871.1150...,376708.472256,87.687580
2,YORK,93077.9,2023,San Bernardino County,POLYGON ((444662.0717799134 -281943.4417934166...,84711.608635,91.011517
3,KELLY,89326.1,2023,Del Norte County,MULTIPOLYGON (((-324518.0573604496 440989.3817...,76804.999267,85.982707
4,MOSQUITO,76771.4,2022,El Dorado County,MULTIPOLYGON (((-59180.38997191671 101054.4453...,13354.760600,17.395489


In [6]:
#aligning county name to simplify df for join between income and fire data 
income_data_clean = (
    income_data
    .assign(county_name = lambda d: d['county'].str.strip() + ' County')
    .drop(columns=[
        'county',
        'GEOID'
    ])
)
income_data_clean.head()


,year,Median_income,Income_MOE,policies,exposure,county_name
0,2022,122488,1231,2956,3047740652,Alameda County
1,2022,101125,17442,263,179049160,Alpine County
2,2022,74853,6048,4041,2808434094,Amador County
3,2022,66085,2261,3722,2472656045,Butte County
4,2022,77526,3875,6983,4970383068,Calaveras County


In [7]:
fire_income_joined = fire_data_clean.merge(
    income_data_clean[['county_name', 'year', 'Median_income','policies','exposure']],
    on=['county_name', 'year'],
    how='left',
)

fire_income_joined.head()


,fire_name,total_fire_acres,year,county_name,geometry,acres_burned,percent_fire_county,Median_income,policies,exposure
0,PARK,429603.0,2024,Butte County,MULTIPOLYGON (((-137757.5718037696 224708.7088...,52896.557968,12.312893,67928,8165,6052068909
1,PARK,429603.0,2024,Tehama County,MULTIPOLYGON (((-162090.1114197266 262871.1150...,376708.472256,87.687580,63784,1498,816861910
2,YORK,93077.9,2023,San Bernardino County,POLYGON ((444662.0717799134 -281943.4417934166...,84711.608635,91.011517,82184,41124,27416056702
3,KELLY,89326.1,2023,Del Norte County,MULTIPOLYGON (((-324518.0573604496 440989.3817...,76804.999267,85.982707,66780,215,107233824
4,MOSQUITO,76771.4,2022,El Dorado County,MULTIPOLYGON (((-59180.38997191671 101054.4453...,13354.760600,17.395489,99246,15454,12343524794


In [8]:
county_fire_summary = (
    fire_income_joined
    #.query('year == 2024') #can querry a specific year if want 
    .groupby(['county_name', 'year'], as_index=False)
    .agg(
        total_acres_burned=('acres_burned', 'sum'),
        number_of_fires=('fire_name', 'nunique'),
        county_median_income=('Median_income', 'first'),
        county_policies=('policies', 'first'),
        county_exposure=('exposure', 'first')
    )
    .round({'total_acres_burned': 2})
    .sort_values('total_acres_burned', ascending=False)
    .reset_index(drop=True)
)

county_fire_summary.head()


,county_name,year,total_acres_burned,number_of_fires,county_median_income,county_policies,county_exposure
0,Tehama County,2024,377010.68,10,63784,1498,816861910
1,Kern County,2024,107882.60,205,70210,8551,4593896136
2,Del Norte County,2023,104548.97,8,66780,215,107233824
3,Siskiyou County,2022,86271.28,21,53898,965,528828514
4,San Bernardino County,2023,85048.91,4,82184,41124,27416056702


From here on down, this will be about visualization and connecting the two datasets 

In [9]:
#loading in shapefiles
fire_shp = raw_fldr / 'cal_fire_historical_data' / 'California_Historic_Fire_Perimeters_-6302641808644124224.zip'
ca_county_boundaries = raw_fldr / 'ca_counties_boundaries.zip'

#create geopandas dataframe
wildfires_gdf = gpd.read_file(fire_shp)
ca_counties_gdf = gpd.read_file(ca_county_boundaries)

#align crs to ensure proper overlap 
wildfires_gdf = wildfires_gdf.to_crs(epsg=3310)
ca_counties_gdf = ca_counties_gdf.to_crs(epsg=3310)

#checking load in 
ca_counties_gdf.head(2)

,CDTFA_COUN,CDTFA_COPR,CENSUS_PLA,CENSUS_GEO,CENSUS_P_1,GNIS_PLACE,GNIS_ID,CDT_COUNTY,PRIMARY_DO,CENSUS_POP,CDT_NAME_S,OFFSHORE,AREA_SQMI,GlobalID,geometry
0,Alameda County,01000,Alameda County,06001,County,Alameda County,1675839,ALA,None,0,Alameda,None,746.727264,d99a13af-9851-4a4d-a926-9df7d5398bc8,"POLYGON ((-136862.653 -21116.482, -136862.975 ..."
1,Alpine County,02000,Alpine County,06003,County,Alpine County,1675840,ALP,None,0,Alpine,None,743.406638,2f9081f0-a1db-4c3d-a0c2-04d8106705a0,"POLYGON ((35904.446 77650.345, 35916.626 77625..."


In [10]:
map_year = 2024

# attach that year's income + fire summary to the county shapes
counties_year = (
    ca_counties_gdf[['CDTFA_COUN', 'geometry']]
    .rename(columns={'CDTFA_COUN': 'county_name'})
    .merge(
        income_data_clean.query('year == @map_year')[['county_name', 'Median_income']],
        on='county_name', 
        how='left'
    )
    .merge(
        county_fire_summary.query('year == @map_year')[['county_name', 'total_acres_burned', 'number_of_fires']],
        on='county_name', 
        how='left'
    )
)

# counties shaded by income: darker = higher
income_fire_map = counties_year.explore(
    column='Median_income',
    cmap='Blues',
    tiles='OpenStreetMap',
    style_kwds={'fillOpacity': 0.6, 'weight': 0.5, 'color': '#000000'},
    tooltip=['county_name', 'Median_income', 'total_acres_burned', 'number_of_fires'],
    legend=True,
    legend_kwds={'caption': f'Median Household Income ({map_year})'},
    name='County Median Income'
)

# that year's fire perimeters on top
wildfires_gdf.query('YEAR_ == @map_year').explore(
    m=income_fire_map,
    color='#e21d1d',
    style_kwds={'fillOpacity': 0.8, 'weight': 0.5},
    tooltip=['FIRE_NAME', 'GIS_ACRES'],
    name=f'{map_year} Fires'
)

folium.LayerControl().add_to(income_fire_map)
income_fire_map
